In [14]:
import analysis_tools
import importlib
import analysis_tools.beam_selection
importlib.reload(analysis_tools.beam_selection)
importlib.reload(analysis_tools)
from analysis_tools import DataLoader
from analysis_tools import BeamSelection, Cut, print_cherenkov_thresholds, SelectionMonitor

### Open the file

In [15]:
run_number = 1928
FILE = f"/eos/experiment/wcte/data/2025_commissioning/processed_offline_data/production_v1_0/{run_number}/WCTE_merged_production_R{run_number}.root"
loader = DataLoader(FILE)


In [16]:
# Check which particles are above Cherenkov threshold in each ACT for this run.
# This tells you whether the ACT-based cuts in your selection are meaningful.
# Look out for kaon runs in particular 
vme_run_info = loader.get_vme_analysis_run_info()
print_cherenkov_thresholds(vme_run_info)

Run momentum : 260 MeV/c
n (act_eveto)  = 1.0100   n (act_tagger) = 1.1100

Particle   Mass [MeV]   Thresh. ACT eveto [MeV]  Above?   Thresh. ACT tagger [MeV]  Above?
------------------------------------------------------------------------------------------
electron        0.511                      3.6     yes                       1.1     yes
muon          105.660                    745.3      no                     219.3     yes
pion          139.570                    984.5      no                     289.7      no
kaon          493.680                   3482.2      no                    1024.7      no
proton        938.270                   6618.0      no                    1947.6      no
deuteron     1876.540                  13236.1      no                    3895.1      no
helium3      2808.390                  19808.9      no                    5829.3      no


In [17]:
vme_scalar_results = loader.get_vme_analysis_scalar_results()

# --- Define your particle selections ---
# Thresholds come from the beam analysis. You can replace any value with your own.
# Call .describe() on any selection to print exactly what cuts are applied.
# Only build the selections you need for your analysis.

# PIONS: fast particles that do not produce Cherenkov light in either ACT.
#   act_eveto_cut  — pions are BELOW this threshold (electrons are above it)
#   act_tagger_cut — pions are BELOW this threshold (muons are above it)
#   proton_tof_cut — pions are fast, i.e. TOF BELOW this value (protons are above it)
pion_sel = BeamSelection.pion(
    act_eveto_cut  = vme_scalar_results['act_eveto_cut'],
    act_tagger_cut = vme_scalar_results['act_tagger_cut'],
    proton_tof_cut = vme_scalar_results['proton_tof_cut'],
)

# MUONS: fast particles, below threshold in the upstream ACT (act_eveto),
#        but ABOVE threshold in the downstream ACT (act_tagger).
#   act_eveto_cut  — muons are BELOW this threshold (electrons are above it)
#   act_tagger_cut — muons are ABOVE this threshold (pions are below it)
#   proton_tof_cut — muons are fast, i.e. TOF BELOW this value
muon_sel = BeamSelection.muon(
    act_eveto_cut  = vme_scalar_results['act_eveto_cut'],
    act_tagger_cut = vme_scalar_results['act_tagger_cut'],
    proton_tof_cut = vme_scalar_results['proton_tof_cut'],
)

# ELECTRONS: fast particles that are ABOVE threshold in the upstream ACT (act_eveto).
#   act_eveto_cut  — electrons are ABOVE this threshold (pions and muons are below it)
#   proton_tof_cut — electrons are fast, i.e. TOF BELOW this value
ele_sel = BeamSelection.electron(
    act_eveto_cut  = vme_scalar_results['act_eveto_cut'],
    proton_tof_cut = vme_scalar_results['proton_tof_cut'],
)

# PROTONS: slow particles identified by their TOF falling in a window ABOVE
#          the fast/slow separation value. Only available when proton_tof_cut > 0.
# proton_sel = BeamSelection.proton(
#     proton_tof_cut    = vme_scalar_results['proton_tof_cut'],
#     proton_tof_window = 30,  # width of the TOF window [ns]
# )

pion_sel.describe()

Selection : pion
  vme_tof_corr           pass all (no TOF separation available for this run)
  vme_act_eveto          < 2.91 PE  [below threshold]
  vme_act_tagger         < 2.83 PE  [below threshold]


### Define the PID selections based on the beam analysis information

### Loading the data in batches

In [ ]:
import time
import awkward as ak

loader.apply_mPMT_data_quality_cuts()
loader.apply_vme_event_quality_cuts()
loader.apply_t5_event_quality_cuts()

# Enable parquet output for the selections you want to save.
# Default filename is "<particle>.parquet". Pass a path to override.
pion_sel.enable_parquet_output(f"run{run_number}_pions.parquet")
muon_sel.enable_parquet_output(f"run{run_number}_muons.parquet")
ele_sel.enable_parquet_output(f"run{run_number}_electrons.parquet")

selections = [pion_sel, muon_sel, ele_sel]
monitor    = SelectionMonitor(selections, update_every=10, vme_run_info=vme_run_info)

start_time = time.time()
n_windows_passing = 0

for i_batch, batch in enumerate(loader.iterate(verbose=False, step_size="100 MB")):
    n_windows_passing += len(batch)
    monitor.update(batch)
    for sel in selections:
        sel._write_to_parquet(batch[sel.mask(batch)])

for sel in selections:
    sel.close_parquet_writer()

monitor.show()
print(f"Loaded {n_windows_passing} events across {i_batch+1} batches  "
      f"({time.time() - start_time:.1f} s)")

In [ ]:
# Read back the saved pion events and inspect them.
# The same pattern works for muons (muon_sel._parquet_path) and electrons.
pions = ak.from_parquet(pion_sel._parquet_path)

print(f"Total pions saved : {len(pions)}")
print(f"Mean corrected TOF: {float(ak.mean(pions['vme_tof_corr'])):.2f} ns")
print(f"Mean act_eveto    : {float(ak.mean(pions['vme_act_eveto'])):.3f} PE")
print(f"Mean act_tagger   : {float(ak.mean(pions['vme_act_tagger'])):.3f} PE")


Get the good PMT list (by mPMT slot and pmt position). Any hit which is not from a good channel is be masked. This is useful for determining which channels are reading out stably for comparison to monte carlo.

In [ ]:
#get the good PMTs slot and position
good_wcte_mpmt_slots, good_wcte_pmt_pos = loader.get_good_wcte_pmts()
print(good_wcte_mpmt_slots,good_wcte_pmt_pos)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 81, 81, 81, 81, 81, 81, 81, 81, 81, 81] [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, ..., 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [ ]:
print("Get the configuration data from the merged file:")
config = loader.get_configuration()
print(config)

print("Get the data quality metrics from the merged file:")
dqm = loader.get_data_quality_metrics()
print(dqm)

print("Get the data quality metrics from the merged file:")
dqm = loader.get_data_quality_metrics()
print(dqm)

print("Get the vme_analysis_scalar_results from the merged file:")
vme_scalar_results = loader.get_vme_analysis_scalar_results()
print(vme_scalar_results)

print("Get the vme_analysis_scalar_results from the merged file:")
vme_run_info = loader.get_vme_analysis_run_info()
print(vme_run_info)

Get the configuration data from the merged file:
{run_configuration: 'Good_mpmt_beam_v47', good_wcte_pmts: [0, ...], ...}
Get the data quality metrics from the merged file:
{n_good_pmt_channels: 1541, n_triggers: 1961864, n_bad_triggers: 0, ...}
Get the data quality metrics from the merged file:
{n_good_pmt_channels: 1541, n_triggers: 1961864, n_bad_triggers: 0, ...}
Get the vme_analysis_scalar_results from the merged file:
{act_eveto_cut: 2.91, act_tagger_cut: 2.83, proton_tof_cut: 0, ...}
Get the vme_analysis_scalar_results from the merged file:
{run_number: 1928, run_momentum: -260, n_eveto: 1.01, n_tagger: 1.11, ...}


### Demonstrate functions to apply the nominal particle identification using the beam monitors 

Apply the nominal PID using the VME time of flight and charge deposited in the ACTs by each particle based on the nominal cut lines identified in the beam analysis code. 

<span style="color:red">This is the nominal particle identification which is *not* optimised for any analysis and can contain issues, please use with care. </span>